<a href="https://colab.research.google.com/github/ancientphoenix34/HG-1/blob/main/hugging_face.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:

# Let's install key libraries
print("Installing necessary libraries...")
# Torch - A deep learning framework used to build and run neural networks.
# bitsandbytes - A library for model optimization using quantization.
!pip install -q transformers accelerate bitsandbytes torch pypdf gradio
print("Libraries installed successfully!")

Installing necessary libraries...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 336.3/336.3 kB 19.8 MB/s eta 0:00:00
Libraries installed successfully!


In [2]:
# Let's import these libraries
import torch  # PyTorch, the backend for transformers
import pypdf  # For reading PDFs
import gradio as gr  # For building the UI
from IPython.display import display, Markdown  # For nicer printing in notebooks
print("Core libraries imported.")

Core libraries imported.


In [3]:
import os
from huggingface_hub import login, notebook_login
print("Attempting Hugging Face login...")

# Use notebook_login() for an interactive prompt in Colab/Jupyter
# This is generally preferred for notebooks.

notebook_login()
print("Login successful (or token already present)!")

Attempting Hugging Face login...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Login successful (or token already present)!


In [4]:
# Check if GPU is available (essential for running these models)
# Why GPU is Important: LLMs involve billions of calculations (matrix multiplications).
# GPUs are designed for massive parallel processing, making these calculations thousands of times faster than a standard CPU.
# Running these models on a CPU would take an impractically long time (hours for a single answer instead of seconds/minutes).
if torch.cuda.is_available():
    print(f"GPU detected: {torch.cuda.get_device_name(0)}")
    # Set default device to GPU
    torch.set_default_device("cuda")
    print("PyTorch default device set to CUDA (GPU).")
else:
    print("WARNING: No GPU detected. Running these models on CPU will be extremely slow!")
    print("Make sure 'GPU' is selected in Runtime > Change runtime type.")

GPU detected: Tesla T4
PyTorch default device set to CUDA (GPU).


In [5]:
# Helper function for markdown display
def print_markdown(text):
    """Displays text as Markdown in Colab/Jupyter."""
    display(Markdown(text))

In [6]:
from transformers import pipeline

# Load a sentiment classifier model on financial news data
# Check the model here: https://huggingface.co/ProsusAI/finbert
pipe = pipeline(model = "ProsusAI/finbert")
#pipe("Tesla stock surged after record-breaking quarterly earnings.")
pipe("Tesla stock surged after record-breaking quarterly earnings.")
#pipe("Microsoft is scheduled to release its earnings report next week.")

config.json:   0%|          | 0.00/758 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/438M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

BertForSequenceClassification LOAD REPORT from: ProsusAI/finbert
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/252 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

[{'label': 'positive', 'score': 0.9457602500915527}]

In [7]:
from transformers import AutoTokenizer

# Load tokenizer for GPT-2
tokenizer = AutoTokenizer.from_pretrained("gpt2")
#tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

# Encode text to token IDs
#tokens = tokenizer("Hello everyone and welcome to LLM and AI Agents Bootcamp")
tokens = tokenizer("Generative AI is transforming the future of work.")
print(tokens['input_ids'])

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

[8645, 876, 9552, 318, 25449, 262, 2003, 286, 670, 13]


In [8]:
# Let's import AutoModelForCausalLM
from transformers import AutoModelForCausalLM, BitsAndBytesConfig

# Let's choose a small, powerful model suitable for Colab.
# Alternatives you could try (might need login/agreement):
# model_id = "unsloth/gemma-3-4b-it-GGUF"
# model_id = "Qwen/Qwen2.5-3B-Instruct"
# model_id = "microsoft/Phi-4-mini-instruct"
model_id = "unsloth/Llama-3.2-3B-Instruct"

In [9]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code = True)
print("Tokenizer loaded successfully.")

config.json:   0%|          | 0.00/890 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/454 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

Tokenizer loaded successfully.


In [10]:
# Let's Load the Model with Quantization

import transformers
print(f"Current transformers version: {transformers.__version__}")
print(f"Loading model: {model_id}")
print("This might take a few minutes, especially the first time...")

# Create BitsAndBytesConfig for 4-bit quantization
quantization_config = BitsAndBytesConfig(load_in_4bit = True,
                                         bnb_4bit_compute_dtype = torch.float16,  # or torch.bfloat16 if available
                                         bnb_4bit_quant_type = "nf4",  # normal float 4 quantization
                                         bnb_4bit_use_double_quant = True  # use nested quantization for more efficient memory usage
                                         )

# Load the model with the quantization config
model = AutoModelForCausalLM.from_pretrained(model_id,
                                             quantization_config = quantization_config,
                                             device_map = "auto",
                                             trust_remote_code = True)

Current transformers version: 5.0.0
Loading model: unsloth/Llama-3.2-3B-Instruct
This might take a few minutes, especially the first time...


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/234 [00:00<?, ?B/s]

In [11]:
prompt = "Explain how Electric Vehicles work in a serious way!"

In [12]:
# Method 1: Let's test the model and Tokenizer using the .generate() method!

# Let's encode the input first
inputs = tokenizer(prompt, return_tensors = "pt")

# Then we will generate the output
outputs = model.generate(**inputs, max_new_tokens = 1000)

response = tokenizer.decode(outputs[0], skip_special_tokens=True)
print_markdown(response)

Explain how Electric Vehicles work in a serious way! 
**Please note: I will not be using any jargon or technical terms that are not commonly understood by a non-expert.**

Electric Vehicles (EVs) are a type of vehicle that uses electricity from a battery to power its electric motor. Here's a simplified explanation of how they work:

**The Main Components:**

1. **Battery:** The battery is the heart of an Electric Vehicle. It's a large container that stores electrical energy. Most EVs use Lithium-Ion batteries, which are similar to those used in laptops and mobile devices.
2. **Electric Motor:** The electric motor is the component that converts the electrical energy from the battery into mechanical energy. It's essentially a high-tech alternator that spins the wheels of the vehicle.
3. **Power Electronics Controller:** This is a sophisticated computer system that manages the flow of electrical energy between the battery, motor, and other components. It's like the "brain" of the vehicle.
4. **Charging System:** Most EVs come with a charging system that allows you to recharge the battery using an external power source, such as a wall socket or a charging station.

**The Process:**

Here's a step-by-step explanation of how an Electric Vehicle works:

1. **You turn the key:** When you start the vehicle, the Power Electronics Controller sends a signal to the motor, telling it to start spinning.
2. **The motor receives power:** The Power Electronics Controller sends electrical energy from the battery to the motor, which then converts it into mechanical energy.
3. **The wheels spin:** As the motor spins, it turns the wheels of the vehicle, propelling it forward.
4. **The battery discharges:** As the motor uses energy from the battery, the battery's state of charge (SOC) decreases.
5. **The Power Electronics Controller regulates:** The controller ensures that the motor receives the right amount of power to maintain the desired speed and efficiency.
6. **The battery is recharged:** When you plug in the vehicle to a charging station or wall socket, the Power Electronics Controller manages the flow of energy, ensuring that the battery is charged safely and efficiently.

**Additional Features:**

Most modern Electric Vehicles come with additional features, such as:

* **Regenerative braking:** This is a system that captures some of the kinetic energy and converts it back into electrical energy, which is stored in the battery.
* **Onboard charger:** This is a system that allows you to charge the battery using an external power source, even when the vehicle is stationary.
* **Advanced battery management:** This is a system that optimizes the battery's state of charge, ensuring that it's always at the right level for optimal performance and efficiency.

**In Summary:**

Electric Vehicles work by using electricity from a battery to power an electric motor, which then propels the vehicle forward. The Power Electronics Controller manages the flow of energy, ensuring that the motor receives the right amount of power. The battery is recharged using an external power source, and the vehicle comes with additional features like regenerative braking and advanced battery management. I hope this explanation helps you understand how Electric Vehicles work!

In [13]:
# Method 2: alternatively, you can create a pipeline that includes your model and tokenizer
# The pipeline wraps tokenization, generation, and decoding

pipe = pipeline("text-generation",
                model = model,
                tokenizer = tokenizer,
                dtype="auto", # Match model dtype
                device_map = "auto" # Ensure pipeline uses the same device mapping
                )


outputs = pipe(prompt,
               max_new_tokens = 1000, # max_new_tokens limits the length of the generated response.
               temperature = 1, # temperature controls randomness (lower = more focused).
               )

# Print the generated text
print_markdown(outputs[0]['generated_text'])

Passing `generation_config` together with generation-related arguments=({'temperature', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=1000) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Explain how Electric Vehicles work in a serious way! Electric vehicles (EVs) are cars that are powered by electricity stored in a battery, rather than gasoline. There are several types of EVs, but I'll cover the most common ones.
**Main Components:**

1.  **Battery:** The battery is the heart of an electric vehicle. It stores energy in the form of chemical bonds between lithium, nickel, and other materials. The battery pack is typically located in the vehicle's floor pan and consists of multiple cells connected together to form a larger battery.
2.  **Electric Motor:** The electric motor is responsible for converting the electrical energy from the battery into mechanical energy. Most EVs use an induction motor, which is an electromagnetic motor that uses electromagnetic fields to generate torque.
3.  **On-Board Charger:** The on-board charger is a device that converts the AC power from the electrical grid into DC power that the battery can use. This is typically done using a wall connector or a charging station.
4.  **Power Control Unit (PCU):** The PCU is a computer system that controls the flow of electricity between the battery, motor, and other components. It manages the charge and discharge of the battery, as well as the motor's speed and torque.
5.  **Charging System:** The charging system consists of a wall connector or charging station that provides AC power to the on-board charger, which then converts it to DC power for the battery.
6.  **Transmission and Drivetrain:** Most EVs use a single-speed or multi-speed transmission, which uses the electric motor as a generator to provide power to the wheels.
7.  **Cooling System:** Electric vehicles often use advanced cooling systems to manage the temperature of the battery, motor, and other components.

**How They Work:**

1.  The vehicle is plugged into a charging station using an AC power cable.
2.  The AC power is converted to DC power by the on-board charger, which is then stored in the battery pack.
3.  The power control unit (PCU) manages the charge and discharge of the battery, as well as the motor's speed and torque.
4.  When the vehicle is in motion, the PCU controls the flow of electricity between the battery, motor, and other components.
5.  The electric motor uses the DC power from the battery to generate torque and propel the vehicle forward.
6.  As the vehicle decelerates, the motor becomes a generator, converting the electrical energy back into mechanical energy to slow the vehicle down.
7.  The cooling system helps to manage the temperature of the battery, motor, and other components to ensure optimal performance and longevity.

**Benefits:**

*   **Zero Emissions:** Electric vehicles produce no tailpipe emissions, reducing greenhouse gas emissions and air pollution in urban areas.
*   **Lower Operating Costs:** Electric vehicles are generally cheaper to run, with lower fuel costs (electricity is often less expensive than gasoline) and lower maintenance costs (fewer moving parts means less wear and tear).
*   **Improved Performance:** Electric vehicles can accelerate more quickly and have smoother, quieter operation than gasoline-powered vehicles.
*   **Sustainability:** Electric vehicles are often built with sustainable materials and can be designed to be easily recyclable, reducing electronic waste and promoting a more circular economy.

**Challenges:**

*   **Range Anxiety:** Electric vehicles have limited range, which can be a concern for drivers who need to travel long distances without access to charging infrastructure.
*   **Charging Infrastructure:** The development of charging infrastructure is still in its early stages, and many areas lack the necessary charging stations.
*   **High Upfront Costs:** Electric vehicles are often more expensive than gasoline-powered vehicles, making them less accessible to some consumers.
*   **Energy Efficiency:** Electric vehicles require energy to generate the electricity for the battery, which can be a concern for those who want to reduce their overall energy consumption.

Overall, electric vehicles offer a promising alternative to traditional gasoline-powered vehicles, with benefits such as zero emissions, lower operating costs, and improved performance. While there are still challenges to be addressed, the development of electric vehicles is an exciting and rapidly evolving field that holds great potential for a more sustainable and environmentally friendly transportation future.

In [14]:
import requests
from pathlib import Path

# --- Get the PDF File ---
# The previous URL was returning a 403 Forbidden error.
# Using a publicly accessible PDF from arXiv.
pdf_url = "https://arxiv.org/pdf/2307.09288.pdf"
pdf_filename = "LLM_research_paper.pdf"
pdf_path = Path(pdf_filename)

# Download the file if it doesn't exist
if not pdf_path.exists():
    response = requests.get(pdf_url)
    response.raise_for_status()  # Check for download errors
    pdf_path.write_bytes(response.content)
    print(f"PDF downloaded successfully to {pdf_path}")
else:
    print(f"PDF file already exists at {pdf_path}")


# --- Read Text from PDF using pypdf ---
pdf_text = ""

print(f"Reading text from {pdf_path}...")
reader = pypdf.PdfReader(pdf_path)
num_pages = len(reader.pages)
print(f"PDF has {num_pages} pages.")

# Extract text from each page
all_pages_text = []
for i, page in enumerate(reader.pages):

    page_text = page.extract_text()
    if page_text:  # Only add if text extraction was successful
        all_pages_text.append(page_text)
    # print(f"Read page {i+1}/{num_pages}") # Uncomment for progress

# Join the text from all pages
pdf_text = "\n".join(all_pages_text)
print(f"Successfully extracted text. Total characters: {len(pdf_text)}")

PDF downloaded successfully to LLM_research_paper.pdf
Reading text from LLM_research_paper.pdf...
PDF has 77 pages.
Successfully extracted text. Total characters: 260529


In [15]:
# Display a small snippet of the PDF
print("\n--- Snippet of Extracted Text ---")
print_markdown(f"{pdf_text[:1000]}")


--- Snippet of Extracted Text ---


Llama 2: Open Foundation and Fine-Tuned Chat Models
Hugo Touvron∗ Louis Martin† Kevin Stone†
Peter Albert Amjad Almahairi Yasmine Babaei Nikolay Bashlykov Soumya Batra
Prajjwal Bhargava Shruti Bhosale Dan Bikel Lukas Blecher Cristian Canton Ferrer Moya Chen
Guillem Cucurull David Esiobu Jude Fernandes Jeremy Fu Wenyin Fu Brian Fuller
Cynthia Gao Vedanuj Goswami Naman Goyal Anthony Hartshorn Saghar Hosseini Rui Hou
Hakan Inan Marcin Kardas Viktor Kerkez Madian Khabsa Isabel Kloumann Artem Korenev
Punit Singh Koura Marie-Anne Lachaux Thibaut Lavril Jenya Lee Diana Liskovich
Yinghai Lu Yuning Mao Xavier Martinet Todor Mihaylov Pushkar Mishra
Igor Molybog Yixin Nie Andrew Poulton Jeremy Reizenstein Rashi Rungta Kalyan Saladi
Alan Schelten Ruan Silva Eric Michael Smith Ranjan Subramanian Xiaoqing Ellen Tan Binh Tang
Ross Taylor Adina Williams Jian Xiang Kuan Puxin Xu Zheng Yan Iliyan Zarov Yuchen Zhang
Angela Fan Melanie Kambadur Sharan Narang Aurelien Rodriguez Robert Stojnic
Sergey Edunov

In [16]:
# Define a limit for the context length to avoid overwhelming the model

MAX_CONTEXT_CHARS = 6000

def answer_question_from_pdf(document_text, question, llm_pipeline):
    """
    Answers a question based on the provided document text using the loaded LLM pipeline.

    Args:
        document_text (str): The text extracted from the PDF.
        question (str): The user's question.
        llm_pipeline (transformers.pipeline): The initialized text-generation pipeline.

    Returns:
        str: The model's generated answer.
    """
    # Truncate context if necessary
    if len(document_text) > MAX_CONTEXT_CHARS:
        print(f"Warning: Document text ({len(document_text)} chars) exceeds limit ({MAX_CONTEXT_CHARS} chars). Truncating.")
        context = document_text[:MAX_CONTEXT_CHARS] + "..."
    else:
        context = document_text

    # Let's define the Prompt Template
    # We instruct the model to use only the provided document.
    # Using a format the model expects (like Phi-3's chat format) can improve results.
    # <|system|> provides context/instructions, <|user|> is the question.
    # Note: Different models might prefer different prompt structures.
    prompt_template = f"""<|system|>
    You are an AI assistant. Answer the following question based *only* on the provided document text. If the answer is not found in the document, say "The document does not contain information on this topic." Do not use any prior knowledge.

    Document Text:
    ---
    {context}
    ---
    <|end|>
    <|user|>
    Question: {question}<|end|>
    <|assistant|>
    Answer:""" # We prompt the model to start generating the answer

    print(f"\n--- Generating Answer for: '{question}' ---")

    # Run Inference on the chosen model
    outputs = llm_pipeline(prompt_template,
                           max_new_tokens = 500,  # Limit answer length
                           do_sample = True,
                           temperature = 0.2,   # Lower temperature for more factual Q&A
                           top_p = 0.9)

    # Let's extract the answer
    # The output includes the full prompt template. We need the text generated *after* it.
    full_generated_text = outputs[0]['generated_text']
    answer_start_index = full_generated_text.find("Answer:") + len("Answer:")
    raw_answer = full_generated_text[answer_start_index:].strip()

    # Sometimes the model might still include parts of the prompt or trail off.
    # Basic cleanup: Find the end-of-sequence token if possible, or just return raw.
    # Phi-3 uses <|end|> or <|im_end|>
    end_token = "<|end|>"
    if end_token in raw_answer:
            raw_answer = raw_answer.split(end_token)[0]

    print("--- Generation Complete ---")
    return raw_answer


In [17]:
# Let's test the function
test_question = "What is this document about?"
generated_answer = answer_question_from_pdf(pdf_text, test_question, pipe)

print("\nTest Question:")
print_markdown(f"**Q:** {test_question}")
print("\nGenerated Answer:")
print_markdown(f"**A:** {generated_answer}")

Passing `generation_config` together with generation-related arguments=({'top_p', 'temperature', 'max_new_tokens', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=500) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



--- Generating Answer for: 'What is this document about?' ---
--- Generation Complete ---

Test Question:


**Q:** What is this document about?


Generated Answer:


**A:** This document is about the development and release of Llama 2, a collection of pretrained and fine-tuned large language models (LLMs) for dialogue use cases.

In [18]:
# Use models known to fit in Colab free tier with 4-bit quantization

available_models = {
    "Llama 3.2": "unsloth/Llama-3.2-3B-Instruct",
    "Microsoft Phi-4 Mini": "microsoft/Phi-4-mini-instruct",
    "Google Gemma 3": "unsloth/gemma-3-4b-it-GGUF"
    }

In [21]:
import gc
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline, BitsAndBytesConfig

# --- Global State ---
current_model_id = None
current_pipeline = None

# Standardize Quantization Config
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True
)

def load_llm_model(model_name):
    """Loads the selected LLM, unloading the previous one correctly."""
    global current_model_id, current_pipeline

    new_model_id = available_models.get(model_name)
    if not new_model_id:
        return "Invalid model selected.", None

    if new_model_id == current_model_id and current_pipeline is not None:
        return f"{model_name} already loaded.", current_pipeline

    print(f"Switching to model: {model_name}...")

    # CORRECT CLEANUP: Access globals explicitly
    current_pipeline = None

    # Use a list of potential global names to clear
    for var in ['model', 'tokenizer', 'pipe']:
        if var in globals():
            del globals()[var]

    torch.cuda.empty_cache()
    gc.collect()
    print("VRAM Cleared.")

    try:
        # Note: GGUF models will fail here unless using a specific GGUF loader
        if "GGUF" in new_model_id:
            return "Error: GGUF format requires llama-cpp-python. Use a standard HF repo.", None

        new_tokenizer = AutoTokenizer.from_pretrained(new_model_id, trust_remote_code=True)
        new_model = AutoModelForCausalLM.from_pretrained(
            new_model_id,
            quantization_config=quant_config,
            device_map="auto",
            trust_remote_code=True
        )

        new_pipe = pipeline(
            "text-generation",
            model=new_model,
            tokenizer=new_tokenizer
        )

        current_model_id = new_model_id
        current_pipeline = new_pipe
        return f"{model_name} loaded successfully!", current_pipeline

    except Exception as e:
        return f"Error loading {model_name}: {str(e)}", None

In [20]:
# --- Function to handle Q&A Submission ---
# This function now relies on the globally managed 'current_pipeline'
# In a more robust Gradio app, you'd pass the pipeline via gr.State
def handle_submit(question):
    """Handles the user submitting a question."""
    if not current_pipeline:
        return "Error: No model is currently loaded. Please select a model."
    if not pdf_text:
        return "Error: PDF text is not loaded. Please run Section 4."
    if not question:
        return "Please enter a question."

    print(f"Handling submission for question: '{question}' using {current_model_id}")
    # Call the Q&A function defined in Section 5
    answer = answer_question_from_pdf(pdf_text, question, current_pipeline)
    return answer



In [ ]:

# --- Build Gradio Interface using Blocks ---
print("Building Gradio interface...")
with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown(
        f"""
    # PDF Q&A Bot Using Hugging Face Open-Source Models
    Ask questions about the document ('{pdf_filename}' if loaded, {len(pdf_text)} chars).
    Select an open-source LLM to answer your question.
    **Note:** Switching models takes time as the new model needs to be downloaded and loaded into the GPU.
    """
    )

    # Store the pipeline in Gradio state for better practice (optional for this simple version)
    # llm_pipeline_state = gr.State(None)

    with gr.Row():
        model_dropdown = gr.Dropdown(
            choices=list(available_models.keys()),
            label="🤖 Select LLM Model",
            value=list(available_models.keys())[0],  # Default to the first model
        )
        status_textbox = gr.Textbox(label="Model Status", interactive=False)

    question_textbox = gr.Textbox(
        label="❓ Your Question", lines=2, placeholder="Enter your question about the document here..."
    )
    submit_button = gr.Button("Submit Question", variant="primary")
    answer_textbox = gr.Textbox(label="💡 Answer", lines=5, interactive=False)

    # --- Event Handlers ---
    # When the dropdown changes, load the selected model
    model_dropdown.change(
        fn = load_llm_model,
        inputs = [model_dropdown],
        outputs = [status_textbox],  # Update status text. Ideally also update a gr.State for the pipeline
        # outputs=[status_textbox, llm_pipeline_state] # If using gr.State
    )

    # When the button is clicked, call the submit handler
    submit_button.click(
        fn = handle_submit,
        inputs = [question_textbox],
        outputs = [answer_textbox],
        # inputs=[question_textbox, llm_pipeline_state], # Pass state if using it
    )

    # --- Initial Model Load ---
    # Easier: Manually load first model *before* launching Gradio for simplicity here
    initial_model_name = list(available_models.keys())[0]
    print(f"Performing initial load of default model: {initial_model_name}...")
    status, _ = load_llm_model(initial_model_name)
    status_textbox.value = status  # Set initial status
    print("Initial load complete.")


# --- Launch the Gradio App ---
print("Launching Gradio demo...")
demo.launch(debug=True)  # debug=True provides more detailed logs

Building Gradio interface...
Performing initial load of default model: Llama 3.2...
Switching to model: Llama 3.2...


/tmp/ipykernel_1086/885397300.py:3: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as demo:


VRAM Cleared.


Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

Initial load complete.
Launching Gradio demo...
It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://2277ef7165c6285e31.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Both `max_new_tokens` (=500) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Handling submission for question: 'explain human heart in 25 words' using unsloth/Llama-3.2-3B-Instruct

--- Generating Answer for: 'explain human heart in 25 words' ---
--- Generation Complete ---
